## 第16章 SQLite数据库


### 1.创建数据库

- 连接数据库：`conn = sqlite3.connect("sqlite.db", isolation_level=None)`，如果数据库不存在，会自动创建。`isolation_level=None`表示不使用事务。
- 执行SQL语句：`conn.execute(sql)`。
- 关闭连接：`conn.close()`。

- SQLite支持的数据类型：
  - INT 或 INTEGER：整数，类似int。
  - REAL：浮点数，类似float。
  - TEXT：字符串，类似str。
  - BLOB：二进制数据，类似bytes。
  - NULL：空值，类似None。
- 创建表时如果使用关键字 `STRICT`，则启用严格模式，必须给每列指定数据类型，并且插入的数据类型必须正确(SQLite3.37.0开始支持)。

In [ ]:
import sqlite3
conn = sqlite3.connect("res/example.db", isolation_level=None)  # 创建数据库连接
conn.execute("CREATE TABLE IF NOT EXISTS cats (name TEXT NOT NULL, birthdate TEXT, fur TEXT, weight REAL) STRICT")  # 创建表
conn.execute("SELECT name FROM sqlite_schema WHERE type= 'table'").fetchall()  # 查询数据库中所有表
conn.execute("PRAGMA TABLE_INFO(cats)").fetchall()  # 查询表结构
conn.close()  # 关闭数据库连接

### 2.数据库操作CRUD

- 创建数据(C)：`INSERT INTO table VALUES (?, ?, ?, ...), [value1, value2, value3, ...]`。
- 读取数据(R)：`SELECT * FROM table WHERE condition ORDER BY column LIMIT number`。
    - 创建索引：`CREATE INDEX index_name ON table (column)`。
    - 删除索引：`DROP INDEX index_name`。
- 更新数据(U)：`UPDATE table SET column = value WHERE condition`。
- 删除数据(D)：`DELETE FROM table WHERE condition`。

In [ ]:
import sqlite3

# 创建数据库连接
conn = sqlite3.connect("res/cats.db", isolation_level=None)

# 创建数据
conn.execute("INSERT INTO cats VALUES (?, ?, ?, ?)", ["Zophine", "2021-01-24", "black", 5.6])

# 读取数据
for row in conn.execute("SELECT rowid, * FROM cats WHERE fur=? OR birthdate>=? ORDER BY fur ASC, birthdate DESC LIMIT 10", ["black", "2024-01-01"]):
    print("Row data: ", row)

# 更新数据
conn.execute("UPDATE cats SET fur=? WHERE rowid=?", ["orange", 1])

# 删除数据
conn.execute("DELETE FROM cats WHERE rowid>? and rowid<?", [100, 105])

# 关闭数据库连接
conn.close()

### 3.回滚事务

- 开启事务：`conn.execute('BEGIN')`。
- 提交事务：`conn.commit()`。
- 回滚事务：`conn.rollback()`。

### 4.备份数据库

- 备份数据库：`conn.backup(backup_conn)`。

### 5.修改表

- 修改表名：`conn.execute("ALTER TABLE old_table_name RENAME TO new_table_name")`。
- 修改列名：`conn.execute("ALTER TABLE table_name RENAME COLUMN old_column_name TO new_column_name")`。
- 添加新列：`conn.execute("ALTER TABLE table_name ADD COLUMN new_column_name type")`。
- 删除某列：`conn.execute("ALTER TABLE table_name DROP COLUMN column_name")`。
- 删除整表：`conn.execute("DROP TABLE table_name")`。

In [ ]:
import sqlite3
from pathlib import Path

# 备份数据库
conn_ori = sqlite3.connect("res/cats.db", isolation_level=None)
conn = sqlite3.connect("res/cats_backup.db", isolation_level=None)
conn_ori.backup(conn)
conn_ori.close()

# 修改表名
conn.execute("ALTER TABLE cats RENAME TO felines")
print("Tables: ", conn.execute("SELECT name FROM sqlite_schema WHERE type= 'table'").fetchall())

# 修改列名
conn.execute("ALTER TABLE felines RENAME COLUMN fur TO description")
print("Columns: ", conn.execute("PRAGMA table_info(felines)").fetchall())

# 添加新列
conn.execute("ALTER TABLE felines ADD COLUMN is_love INTEGER DEFAULT 1")
print("Columns: ", conn.execute("PRAGMA table_info(felines)").fetchall())

# 删除列
conn.execute("ALTER TABLE felines DROP COLUMN is_love")
print("Columns: ", conn.execute("PRAGMA table_info(felines)").fetchall())

# 删除表
conn.execute("DROP TABLE felines")
print("Tables: ", conn.execute("SELECT name FROM sqlite_schema WHERE type= 'table'").fetchall())

# 关闭数据库连接并删除备份文件
conn.close()
Path("res/cats_backup.db").unlink()

### 6.使用外键

- 启用外键功能：`conn.execute("PRAGMA foreign_keys = ON")`。
- 创建外键：`CREATE TABLE IF NOT EXISTS table_name (column_name, ..., FROEIGN KEY(column_name) REFERENCES parent_table(column_name))`。

### 7.内存数据库

- 创建内存数据库：`sqlite3.connect(':memory:', isolation_level=None)`。

### 8.复制数据库

- 生成整个数据库SQL语句：`conn.iterdump()`。